# Classifier Validation — Hand-coded vs. LLM

This notebook validates the LLM classifications against a hand-coded random sample (`random_sample_handcode.csv`, N=50).

## Strategies compared

| Label | Classifier file | Labels used |
|---|---|---|
| **Strategy A** | `tc_resolutions_fulltext_data_11May26_output.csv` | 3-label: CENTRAL / PERIPHERAL / ABSENT · POLITICAL / AMBIGUOUS / ROUTINE |
| **Strategy B** | *(set in config below)* | 2-label: CENTRAL / ABSENT · POLITICAL / ROUTINE |

## Notebook structure

1. **Setup** — imports and file paths  
2. **Strategy A · Load & repair** — fix LLM PARSE_ERRORs  
3. **Validation A1** — Strategy A vs. hand-coded (original 3 labels)  
4. **Validation A2** — Strategy A vs. hand-coded (binary recoding: PERIPHERAL → ABSENT, AMBIGUOUS → ROUTINE)  
5. **Strategy B · Load** — native binary classifier  
6. **Validation B** — Strategy B vs. hand-coded (binary recoding, same as A2 ground truth)

## 1 · Setup — imports and file paths

In [3]:
import pandas as pd
import json
import re

In [226]:
VALIDATION_PATH        = "tc_resolutions_fulltext_data_27May26_output_gptoss.csv" #"random_sample_handcode_150.csv" #"tc_resolutions_fulltext_data_11May26_output.csv" #"tc_resolutions_fulltext_data_27May26_output_gptoss.csv" #"tc_resolutions_fulltext_data_27May26_output_gptoss.csv" #"random_sample_handcode_150_v2.csv"
CLASSIFIER_PATH        =  "tc_resolutions_fulltext_data_28May28_output_mistral_3label.csv" #"tc_resolutions_fulltext_data_27May26_output_gptoss.csv" #"tc_resolutions_fulltext_data_28May28_output_mistral_3label.csv" #"tc_resolutions_fulltext_data_11May26_output.csv"   # Strategy A (3-label)
BINARY_CLASSIFIER_PATH = "tc_resolutions_fulltext_data_21May26_output_debug_binary.csv"    # Strategy B (2-label) — update path before running Section 5

## 2 · Strategy A — Load & fix PARSE_ERRORs (3-label classifier)

In [227]:
## Fix PARSE_ERRORs

def recover_label(raw: str, key: str, valid: set) -> str:
    """
    Re-parse a raw LLM response that failed the original JSON parser.
    Handles markdown code fences (```json ... ```) and stray whitespace.
    Falls back to regex if json.loads still fails.
    """
    if not isinstance(raw, str):
        return "PARSE_ERROR"
    # Strip markdown fences and whitespace
    cleaned = re.sub(r"```(?:json)?", "", raw).strip().rstrip("`").strip()
    # Some responses are truncated — try to close an open JSON object
    if cleaned.startswith("{") and not cleaned.endswith("}"):
        cleaned += '"}'
    try:
        data = json.loads(cleaned)
        label = data.get(key, "PARSE_ERROR")
        return label if label in valid else "PARSE_ERROR"
    except Exception:
        pass
    # Last resort: regex search for the key
    match = re.search(rf'"{key}"\s*:\s*"([^"]+)"', raw)
    if match:
        label = match.group(1)
        return label if label in valid else "PARSE_ERROR"
    return "PARSE_ERROR"


dcoded = pd.read_csv(CLASSIFIER_PATH)

VALID_TERR = {"CENTRAL", "PERIPHERAL", "ABSENT"}
VALID_POL  = {"POLITICAL", "AMBIGUOUS", "ROUTINE"}

terr_errors_before = (dcoded["territorial"] == "PARSE_ERROR").sum()
pol_errors_before  = (dcoded["politization"] == "PARSE_ERROR").sum()

# Recover territorial labels
mask_terr = dcoded["territorial"] == "PARSE_ERROR"
dcoded.loc[mask_terr, "territorial"] = dcoded.loc[mask_terr, "territorial_reason"].apply(
    lambda r: recover_label(r, "territorial", VALID_TERR)
)

# Recover politization labels
mask_pol = dcoded["politization"] == "PARSE_ERROR"
dcoded.loc[mask_pol, "politization"] = dcoded.loc[mask_pol, "politization_reason"].apply(
    lambda r: recover_label(r, "politization", VALID_POL)
)

terr_errors_after = (dcoded["territorial"] == "PARSE_ERROR").sum()
pol_errors_after  = (dcoded["politization"] == "PARSE_ERROR").sum()

print(f"Territorial  PARSE_ERROR: {terr_errors_before} → {terr_errors_after} remaining")
print(f"Politization PARSE_ERROR: {pol_errors_before}  → {pol_errors_after} remaining")
dcoded[["ID_PAT", "territorial", "politization"]].head()

Territorial  PARSE_ERROR: 23445 → 0 remaining
Politization PARSE_ERROR: 23445  → 0 remaining


,ID_PAT,territorial,politization
0,AUTO 10/1996,PERIPHERAL,AMBIGUOUS
1,AUTO 10/1998,ABSENT,AMBIGUOUS
2,AUTO 10/2013,CENTRAL,POLITICAL
3,AUTO 100/1991,CENTRAL,AMBIGUOUS
4,AUTO 100/2010,CENTRAL,POLITICAL


## 3 · Validation A1 — Strategy A vs. hand-coded (3 labels)

In [229]:
## Validation against hand-coded labels

hand = pd.read_csv(VALIDATION_PATH, #sep=";",
                   encoding="latin1",   # or "iso-8859-1"
                   usecols=["ID_PAT", "territorial", "politization"])
hand = hand.rename(columns={"territorial": "terr_hand", "politization": "pol_hand"})

llm = dcoded[["ID_PAT", "territorial", "politization"]].rename(
    columns={"territorial": "terr_llm", "politization": "pol_llm"}
)

df = hand.merge(llm, on="ID_PAT", how="inner")
print(f"Matched cases: {len(df)}\n")

# Agreement rates
df["agree_terr"] = df["terr_llm"] == df["terr_hand"]
df["agree_pol"]  = df["pol_llm"]  == df["pol_hand"]
print(f"Territorial  agreement: {df['agree_terr'].sum()}/{len(df)}  ({df['agree_terr'].mean():.1%})")
print(f"Politization agreement: {df['agree_pol'].sum()}/{len(df)}  ({df['agree_pol'].mean():.1%})")

Matched cases: 23445

Territorial  agreement: 22108/23445  (94.3%)
Politization agreement: 20178/23445  (86.1%)


### Precision, Recall, F1

In [230]:
from sklearn.metrics import classification_report

# Drop any rows where either label is missing
valid = df.dropna(subset=["terr_hand", "terr_llm", "pol_hand", "pol_llm"])

print("── Territorial ──")
print(classification_report(valid["terr_hand"], valid["terr_llm"],
                             labels=["CENTRAL", "PERIPHERAL", "ABSENT"],
                             zero_division=0))

print("── Politization ──")
print(classification_report(valid["pol_hand"], valid["pol_llm"],
                             labels=["POLITICAL", "AMBIGUOUS", "ROUTINE"],
                             zero_division=0))

── Territorial ──
              precision    recall  f1-score   support

     CENTRAL       0.92      0.77      0.84      1609
  PERIPHERAL       0.14      0.54      0.22       306
      ABSENT       0.99      0.96      0.98     21530

    accuracy                           0.94     23445
   macro avg       0.68      0.76      0.68     23445
weighted avg       0.97      0.94      0.96     23445

── Politization ──
              precision    recall  f1-score   support

   POLITICAL       0.78      0.83      0.80      2241
   AMBIGUOUS       0.71      0.87      0.78      6560
     ROUTINE       0.97      0.86      0.91     14644

    accuracy                           0.86     23445
   macro avg       0.82      0.85      0.83     23445
weighted avg       0.88      0.86      0.87     23445



### Confusion matrices

In [216]:
print("── Territorial (rows = LLM, columns = hand-coded) ──")
display(pd.crosstab(df["terr_llm"], df["terr_hand"],
                    rownames=["LLM"], colnames=["hand-coded"], margins=True))

print("\n── Politization (rows = LLM, columns = hand-coded) ──")
display(pd.crosstab(df["pol_llm"], df["pol_hand"],
                    rownames=["LLM"], colnames=["hand-coded"], margins=True))

── Territorial (rows = LLM, columns = hand-coded) ──


hand-coded,ABSENT,CENTRAL,PERIPHERAL,All
LLM,,,,
ABSENT,105,5,9,119
CENTRAL,2,26,2,30
PERIPHERAL,1,0,0,1
All,108,31,11,150



── Politization (rows = LLM, columns = hand-coded) ──


hand-coded,AMBIGUOUS,POLITICAL,ROUTINE,All
LLM,,,,
AMBIGUOUS,4,3,13,20
POLITICAL,9,10,11,30
ROUTINE,6,4,90,100
All,19,17,114,150


### Disagreements

In [217]:
reason_cols = [c for c in dcoded.columns if c.endswith("_reason")]
detail = df.merge(dcoded[["ID_PAT"] + reason_cols], on="ID_PAT", how="left")

mask = ~df["agree_terr"] | ~df["agree_pol"]
cols = ["ID_PAT", "terr_hand", "terr_llm", "pol_hand", "pol_llm"] + reason_cols
print(f"Cases with at least one disagreement: {mask.sum()}")
detail[mask][cols].reset_index(drop=True)

Cases with at least one disagreement: 54


,ID_PAT,terr_hand,terr_llm,pol_hand,pol_llm,territorial_reason,politization_reason
0,AUTO 215/1985,ABSENT,ABSENT,AMBIGUOUS,ROUTINE,The dispute concerns gender wage equality in l...,The case concerns a specific labor dispute ove...
1,SENTENCIA 52/1999,CENTRAL,ABSENT,ROUTINE,ROUTINE,The dispute centers on procedural recusal and ...,The dispute concerns a procedural recusation a...
2,AUTO 290/1990,ABSENT,ABSENT,AMBIGUOUS,ROUTINE,The dispute concerns procedural notification a...,The dispute concerns a procedural notification...
3,SENTENCIA 41/1987,ABSENT,ABSENT,POLITICAL,ROUTINE,The dispute concerns procedural notification a...,The dispute concerns a private procedural issu...
4,SENTENCIA 40/1995,ABSENT,ABSENT,AMBIGUOUS,ROUTINE,The dispute concerns the legality of a strike ...,The case concerns an individual labor dispute ...
5,SENTENCIA 24/1995,CENTRAL,ABSENT,POLITICAL,ROUTINE,The dispute centers on procedural admissibilit...,The case concerns an individual military disci...
6,AUTO 265/2023,PERIPHERAL,CENTRAL,AMBIGUOUS,POLITICAL,The core issue is whether Aragon's decree on p...,The case challenges a regional emergency procu...
7,SENTENCIA 306/1994,PERIPHERAL,ABSENT,ROUTINE,ROUTINE,The dispute centers on the legality of adminis...,The case concerns an individual’s administrati...
8,AUTO 4/2008,ABSENT,ABSENT,ROUTINE,POLITICAL,The dispute concerns freedom of expression and...,The case involves terrorism‑related speech and...
9,SENTENCIA 68/2012,ABSENT,ABSENT,ROUTINE,POLITICAL,The dispute centers on criminal sentencing and...,The case concerns the controversial 'Parot' do...


## 4 · Validation A2 — Strategy A with binary recoding

Original `df` is untouched. `df_bin` is a separate copy with:
- Territorial: `PERIPHERAL` → `ABSENT`  (only `CENTRAL` = positive)
- Politization: `AMBIGUOUS`  → `ROUTINE` (only `POLITICAL` = positive)

`df_bin` is also used as ground truth in Section 6 (Strategy B validation).

In [231]:
df_bin = df.copy()

# Recode — only the copies, original columns in df are preserved
for col in ["terr_hand", "terr_llm"]:
    df_bin[col] = df_bin[col].replace("PERIPHERAL", "ABSENT")

for col in ["pol_hand", "pol_llm"]:
    df_bin[col] = df_bin[col].replace("AMBIGUOUS", "ROUTINE")

print("Original label counts:")
print("  terr_hand:", df["terr_hand"].value_counts().to_dict())
print("  pol_hand: ", df["pol_hand"].value_counts().to_dict())
print("\nRecoded label counts:")
print("  terr_hand:", df_bin["terr_hand"].value_counts().to_dict())
print("  pol_hand: ", df_bin["pol_hand"].value_counts().to_dict())

Original label counts:
  terr_hand: {'ABSENT': 21530, 'CENTRAL': 1609, 'PERIPHERAL': 306}
  pol_hand:  {'ROUTINE': 14644, 'AMBIGUOUS': 6560, 'POLITICAL': 2241}

Recoded label counts:
  terr_hand: {'ABSENT': 21836, 'CENTRAL': 1609}
  pol_hand:  {'ROUTINE': 21204, 'POLITICAL': 2241}


In [232]:
df_bin["agree_terr"] = df_bin["terr_llm"] == df_bin["terr_hand"]
df_bin["agree_pol"]  = df_bin["pol_llm"]  == df_bin["pol_hand"]

print(f"Territorial  agreement (binary): {df_bin['agree_terr'].sum()}/{len(df_bin)}  ({df_bin['agree_terr'].mean():.1%})  "
      f"vs. original {df['agree_terr'].sum()}/{len(df)}  ({df['agree_terr'].mean():.1%})")
print(f"Politization agreement (binary): {df_bin['agree_pol'].sum()}/{len(df_bin)}  ({df_bin['agree_pol'].mean():.1%})  "
      f"vs. original {df['agree_pol'].sum()}/{len(df)}  ({df['agree_pol'].mean():.1%})")

Territorial  agreement (binary): 22971/23445  (98.0%)  vs. original 22108/23445  (94.3%)
Politization agreement (binary): 22523/23445  (96.1%)  vs. original 20178/23445  (86.1%)


In [233]:
print("── Territorial binary (rows = LLM, columns = hand-coded) ──")
display(pd.crosstab(df_bin["terr_llm"], df_bin["terr_hand"],
                    rownames=["LLM"], colnames=["hand-coded"], margins=True))

print("\n── Politization binary (rows = LLM, columns = hand-coded) ──")
display(pd.crosstab(df_bin["pol_llm"], df_bin["pol_hand"],
                    rownames=["LLM"], colnames=["hand-coded"], margins=True))

── Territorial binary (rows = LLM, columns = hand-coded) ──


hand-coded,ABSENT,CENTRAL,All
LLM,,,
ABSENT,21734,372,22106
CENTRAL,102,1237,1339
All,21836,1609,23445



── Politization binary (rows = LLM, columns = hand-coded) ──


hand-coded,POLITICAL,ROUTINE,All
LLM,,,
POLITICAL,1857,538,2395
ROUTINE,384,20666,21050
All,2241,21204,23445


In [234]:
valid_bin = df_bin.dropna(subset=["terr_hand", "terr_llm", "pol_hand", "pol_llm"])

print("── Territorial (binary) ──")
print(classification_report(valid_bin["terr_hand"], valid_bin["terr_llm"],
                             labels=["CENTRAL", "ABSENT"],
                             zero_division=0))

print("── Politization (binary) ──")
print(classification_report(valid_bin["pol_hand"], valid_bin["pol_llm"],
                             labels=["POLITICAL", "ROUTINE"],
                             zero_division=0))

── Territorial (binary) ──
              precision    recall  f1-score   support

     CENTRAL       0.92      0.77      0.84      1609
      ABSENT       0.98      1.00      0.99     21836

    accuracy                           0.98     23445
   macro avg       0.95      0.88      0.91     23445
weighted avg       0.98      0.98      0.98     23445

── Politization (binary) ──
              precision    recall  f1-score   support

   POLITICAL       0.78      0.83      0.80      2241
     ROUTINE       0.98      0.97      0.98     21204

    accuracy                           0.96     23445
   macro avg       0.88      0.90      0.89     23445
weighted avg       0.96      0.96      0.96     23445



## 5 · Strategy B — Load native binary classifier

Loads the output of the 2-label classifier (CENTRAL/ABSENT · POLITICAL/ROUTINE).  
Ground truth for this comparison is `df_bin` from Section 4 (same binary recoding).

In [63]:
clf_b = pd.read_csv(BINARY_CLASSIFIER_PATH,
                    usecols=["ID_PAT", "territorial", "politization"])
clf_b = clf_b.rename(columns={"territorial": "terr_b", "politization": "pol_b"})

print(f"Strategy B rows: {len(clf_b):,}  |  unique ID_PAT: {clf_b['ID_PAT'].nunique():,}")
print("\nTerritorial distribution:")
print(clf_b["terr_b"].value_counts().to_string())
print("\nPolitization distribution:")
print(clf_b["pol_b"].value_counts().to_string())
clf_b.head()

Strategy B rows: 100  |  unique ID_PAT: 100

Territorial distribution:
terr_b
ABSENT     96
CENTRAL     4

Politization distribution:
pol_b
ROUTINE      92
POLITICAL     8


,ID_PAT,terr_b,pol_b
0,AUTO 8/1984,ABSENT,ROUTINE
1,AUTO 184/2001,ABSENT,ROUTINE
2,AUTO 42/1985,ABSENT,ROUTINE
3,AUTO 213/1986,ABSENT,ROUTINE
4,AUTO 215/1985,ABSENT,ROUTINE


## 6 · Validation B — Strategy B vs. hand-coded (binary)

Merge Strategy B output with `df_bin` (binary-recoded hand-coded labels from Section 4) and compute the same metrics.

In [124]:
df_b = df_bin[["ID_PAT", "terr_hand", "pol_hand"]].merge(
    clf_b[["ID_PAT", "terr_b", "pol_b"]],
    on="ID_PAT", how="inner"
)
print(f"Matched cases: {len(df_b)}\n")

df_b["agree_terr"] = df_b["terr_b"] == df_b["terr_hand"]
df_b["agree_pol"]  = df_b["pol_b"]  == df_b["pol_hand"]
print(f"Territorial  agreement: {df_b['agree_terr'].sum()}/{len(df_b)}  ({df_b['agree_terr'].mean():.1%})")
print(f"Politization agreement: {df_b['agree_pol'].sum()}/{len(df_b)}  ({df_b['agree_pol'].mean():.1%})")

Matched cases: 100

Territorial  agreement: 94/100  (94.0%)
Politization agreement: 89/100  (89.0%)


In [109]:
print("── Territorial binary (rows = LLM, columns = hand-coded) ──")
display(pd.crosstab(df_b["terr_b"], df_b["terr_hand"],
                    rownames=["LLM"], colnames=["hand-coded"], margins=True))

print("\n── Politization binary (rows = LLM, columns = hand-coded) ──")
display(pd.crosstab(df_b["pol_b"], df_b["pol_hand"],
                    rownames=["LLM"], colnames=["hand-coded"], margins=True))

── Territorial binary (rows = LLM, columns = hand-coded) ──


hand-coded,ABSENT,CENTRAL,All
LLM,,,
ABSENT,91,5,96
CENTRAL,1,3,4
All,92,8,100



── Politization binary (rows = LLM, columns = hand-coded) ──


hand-coded,POLITICAL,ROUTINE,All
LLM,,,
POLITICAL,3,5,8
ROUTINE,6,86,92
All,9,91,100


In [110]:
valid_b = df_b.dropna(subset=["terr_hand", "terr_b", "pol_hand", "pol_b"])

print("── Territorial (binary) ──")
print(classification_report(valid_b["terr_hand"], valid_b["terr_b"],
                             labels=["CENTRAL", "ABSENT"],
                             zero_division=0))

print("── Politization (binary) ──")
print(classification_report(valid_b["pol_hand"], valid_b["pol_b"],
                             labels=["POLITICAL", "ROUTINE"],
                             zero_division=0))

── Territorial (binary) ──
              precision    recall  f1-score   support

     CENTRAL       0.75      0.38      0.50         8
      ABSENT       0.95      0.99      0.97        92

    accuracy                           0.94       100
   macro avg       0.85      0.68      0.73       100
weighted avg       0.93      0.94      0.93       100

── Politization (binary) ──
              precision    recall  f1-score   support

   POLITICAL       0.38      0.33      0.35         9
     ROUTINE       0.93      0.95      0.94        91

    accuracy                           0.89       100
   macro avg       0.65      0.64      0.65       100
weighted avg       0.88      0.89      0.89       100

